# 🎬 Zodiel – Avatar parlante (voce addestrata)
Testo → **tua voce addestrata** → labiale sul **video senza parlare** → video finito.

**Prima di iniziare**
1. Settings → **GPU T4 x2** e **Internet on**.
2. In "Input" devono esserci **solo** questi due dataset:
   - **voce zodiel modello** (lora_epoca04.pt, riferimento.wav, zodiel_voce.py)
   - il dataset con il **nuovo video senza parlare**

Esegui le celle **una alla volta, in ordine**. La prima volta in ogni sessione le installazioni richiedono circa 15 minuti.

## 1. Installazione voce (2–3 minuti)

In [ ]:
!pip install -q chatterbox-tts peft
!pip uninstall -y -q torchvision
print('Voce: strumenti pronti ✔')

## 2. Installazione labiale LatentSync (10–15 minuti)

In [ ]:
%%bash
mkdir -p /kaggle/temp && cd /kaggle/temp
fallito() { echo; echo "❌ ERRORE: $1"; exit 1; }
[ -d LatentSync ] || git clone -q --depth 1 https://github.com/bytedance/LatentSync.git || fallito clone
cd LatentSync
sed -i 's/get_device_capability()\[0\] > 7/get_device_capability()[0] >= 7/' scripts/inference.py
pip install -q uv
[ -x /kaggle/temp/lsenv/bin/python ] || uv venv -q --seed --python 3.10 /kaggle/temp/lsenv || fallito python
PY=/kaggle/temp/lsenv/bin/python
grep -v -i gradio requirements.txt > /kaggle/temp/req_ls.txt
$PY -m pip install -q cython numpy==1.26.4 || fallito librerie
$PY -m pip install -q -r /kaggle/temp/req_ls.txt || fallito librerie
HF=/kaggle/temp/lsenv/bin/huggingface-cli
[ -f checkpoints/whisper/tiny.pt ] || $HF download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints > /dev/null 2>&1 || fallito modelli
[ -f checkpoints/latentsync_unet.pt ] || $HF download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints > /dev/null 2>&1 || fallito modelli
echo "✅ LatentSync pronto"

## 3. ⚙️ Il testo da far dire all'avatar
Scrivilo qui. Usa la punteggiatura per le pause: virgole brevi, punti lunghi.

In [ ]:
TESTO = """
Ciao, e benvenuti. Oggi voglio raccontarvi una carta molto speciale: la Stella.
È la carta della speranza, quella che arriva dopo un periodo difficile.
Quando esce in una lettura, ci invita a fermarci, a respirare, e a fidarci di nuovo della vita.
Non promette miracoli, ma ricorda che dopo la tempesta torna sempre il sereno. A presto!
"""
# regolazioni della voce
ESPRESSIVITA = 0.5   # più alto = più espressiva
RITMO = 0.4          # più basso = più lenta e posata
print('Testo salvato ✔')

## 4. Voce addestrata (1–3 minuti)

In [ ]:
import sys, re, gc, numpy as np, torch, torchaudio
from pathlib import Path
from IPython.display import Audio, display
import peft.import_utils as _iu, peft.tuners.lora.torchao as _lt
_iu.is_torchao_available = lambda: False
_lt.is_torchao_available = lambda: False

MOD = next(Path('/kaggle/input').rglob('lora_epoca04.pt')).parent
sys.path.insert(0, str(MOD))
import zodiel_voce as zv
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

tts = ChatterboxMultilingualTTS.from_pretrained(device='cuda')
zv.carica_lora(tts.t3, MOD / 'lora_epoca04.pt')        # applica la tua voce addestrata
RIF = str(MOD / 'riferimento.wav')

frasi = [f.strip() for f in re.split(r'(?<=[.!?])\s+', ' '.join(TESTO.split())) if f.strip()]
pausa = np.zeros(int(0.35 * tts.sr), dtype=np.float32)
pezzi = []
for i, f in enumerate(frasi, 1):
    print(f'{i}/{len(frasi)}:', f)
    torch.manual_seed(42 + i)
    w = tts.generate(f, language_id='it', audio_prompt_path=RIF, exaggeration=ESPRESSIVITA, cfg_weight=RITMO)
    pezzi += [w.squeeze().cpu().numpy(), pausa]
audio = np.concatenate(pezzi)
VOCE = '/kaggle/temp/voce.wav'
Path('/kaggle/temp').mkdir(exist_ok=True)
torchaudio.save(VOCE, torch.from_numpy(audio).unsqueeze(0), tts.sr)
torchaudio.save('/kaggle/working/voce.wav', torch.from_numpy(audio).unsqueeze(0), tts.sr)
DURATA = len(audio) / tts.sr
print(f'Durata voce: {DURATA:.1f} s')
display(Audio(VOCE))
del tts; gc.collect(); torch.cuda.empty_cache()

## 5. Preparazione del video senza parlare
Se la voce è più lunga del video, il video riparte dall'inizio con una dissolvenza morbida (mai al contrario).

In [ ]:
import subprocess
VIDEO_EXT = ('.mp4', '.mov', '.m4v', '.mkv', '.webm')
vid = next(p for p in Path('/kaggle/input').rglob('*') if p.suffix.lower() in VIDEO_EXT)
durata_v = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', str(vid)],
                                capture_output=True, text=True).stdout)
print(f'Video base: {vid.name} ({durata_v:.0f} s)')

BASE = '/kaggle/temp/base25.mp4'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(vid), '-r', '25', '-an',
                '-vf', "scale='if(gt(ih,1080),-2,iw)':'if(gt(ih,1080),1080,ih)',format=yuv420p",
                '-c:v', 'libx264', '-crf', '17', BASE], check=True)

# ripetizioni con dissolvenza di 0,6 s finché il video copre la voce
X = 0.6
n = 1
while n * durata_v - (n - 1) * X < DURATA + 1:
    n += 1
if n == 1:
    filtro, ingressi = None, ['-i', BASE]
else:
    ingressi = sum((['-i', BASE] for _ in range(n)), [])
    parti, prec, t = [], '[0:v]', durata_v - X
    for k in range(1, n):
        out = f'[v{k}]'
        parti.append(f'{prec}[{k}:v]xfade=transition=fade:duration={X}:offset={t:.2f}{out}')
        prec, t = out, t + durata_v - X
    filtro = ';'.join(parti)
AVATAR = '/kaggle/temp/avatar.mp4'
cmd = ['ffmpeg', '-y', '-loglevel', 'error'] + ingressi
if filtro:
    cmd += ['-filter_complex', filtro, '-map', prec]
cmd += ['-t', f'{DURATA + 0.5:.2f}', '-r', '25', '-c:v', 'libx264', '-crf', '17', '-pix_fmt', 'yuv420p', AVATAR]
subprocess.run(cmd, check=True)
print(f'Video preparato ✔ ({n} ripetizioni)' if n > 1 else 'Video preparato ✔')

## 6. Labiale con LatentSync (10–20 minuti per 30 secondi)

In [ ]:
!cd /kaggle/temp/LatentSync && MPLBACKEND=Agg CUDA_VISIBLE_DEVICES=1 /kaggle/temp/lsenv/bin/python -m scripts.inference \
    --unet_config_path configs/unet/stage2.yaml \
    --inference_ckpt_path checkpoints/latentsync_unet.pt \
    --inference_steps 20 --guidance_scale 1.5 --enable_deepcache \
    --video_path /kaggle/temp/avatar.mp4 --audio_path /kaggle/temp/voce.wav \
    --temp_dir /kaggle/temp/ls_tmp --video_out_path /kaggle/working/avatar_finale.mp4

from IPython.display import Video
display(Video('/kaggle/working/avatar_finale.mp4', embed=True, width=720))
print('Scaricalo da Output → /kaggle/working → avatar_finale.mp4')